In [ ]:
!pip install transformers bitsandbytes accelerate -q


In [ ]:
import pandas as pd
from google.colab import files


df = pd.read_csv("tickets_with_features.csv")

# Stratified sample: 750 per priority class
df_sample = (
    df.groupby("Priority_Level", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 1250), random_state=42))
    .reset_index(drop=True)
)

print(df_sample["Priority_Level"].value_counts())
print(f"Total sample: {len(df_sample)}")

Priority_Level
Critical    1250
High        1250
Low         1250
Medium      1250
Name: count, dtype: int64
Total sample: 5000


/tmp/ipykernel_11271/3590778436.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 1250), random_state=42))


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded")
print(f"Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded
Memory used: 13.36 GB


In [ ]:
def build_prompt(subject: str, description: str) -> str:
    return f"[INST] You are a support ticket severity classifier.\n\nTicket Subject: {subject}\nTicket Description: {description}\n\nRate the TRUE severity of this ticket on a scale of 1 to 4:\n1 = Low (general inquiry, no urgency)\n2 = Medium (minor issue, some impact)\n3 = High (significant issue, affects workflow)\n4 = Critical (data loss, security breach, system down)\n\nReply with ONLY a single digit (1, 2, 3, or 4). No explanation. [/INST]"


def score_ticket(subject: str, description: str) -> int:
    prompt = build_prompt(subject, description)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    for char in response:
        if char in "1234":
            return int(char)
    return 2

In [ ]:
from tqdm import tqdm

scores = []
failed = 0

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        score = score_ticket(row["Ticket_Subject"], row["Ticket_Description"])
    except Exception:
        score = 2
        failed += 1
    scores.append(score)

df_sample["llm_raw_score"] = scores
print(f"Done. Failed: {failed}")
print(df_sample["llm_raw_score"].value_counts())

100%|██████████| 5000/5000 [1:32:10<00:00,  1.11s/it]

Done. Failed: 0
llm_raw_score
3    2276
1    1133
4     894
2     697
Name: count, dtype: int64


In [ ]:
from scipy.stats import rankdata
import numpy as np

# Normalize raw 1-4 score to 0-1
df_sample["llm_score"] = (df_sample["llm_raw_score"] - 1) / 3.0

# Save only what we need
df_sample[["Ticket_ID", "llm_raw_score", "llm_score"]].to_csv("llm_scores.csv", index=False)

print(df_sample["llm_score"].describe())
files.download("llm_scores.csv")

count    5000.000000
mean        0.528733
std         0.342147
min         0.000000
25%         0.333333
50%         0.666667
75%         0.666667
max         1.000000
Name: llm_score, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>